# 1. Κατεβάζουμε τα δεδομένα

In [17]:
import json
import os
from gdown import download
import torch
def download_data(id, filename):
    if not os.path.exists(filename):
        download(id=id, output=filename, quiet=False)
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)
movies = download_data('1bwVnwW9FL4-pMUpOTzXOomHc150NncmW', 'movies.json')
levels = download_data('1O_vJ7aoWfakDKFCZb0zc51BjzUoLJrCW', 'levels.json')
embeddings = torch.tensor( download_data('1E1lHQQ09yQWSw-YWmsnRt8ukbV7-Gznr', 'embeddings.json') )

# 2. Αρχικοποιούμε το μοντέλο.

Η σελίδα χρησιμοποιεί μια διακριτοποιημένη έκδοση οπότε μπορεί να έχει μικρές διαφορές με την python.

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
def embed_query(query):
    return model.encode([query], convert_to_numpy=False, convert_to_tensor=True)[0]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7529.76it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Διαφορετικά μπορείτε να χρησιμοποιήσετε αυτό το μοντέλο που είναι πιο κοντά στην υλοποίηση της σελίδας

In [19]:
%pip install onnxruntime
import onnxruntime as ort
from transformers import AutoTokenizer
import transformers
import torch
import requests

url = "https://huggingface.co/Xenova/multilingual-e5-small/resolve/main/onnx/model_quantized.onnx"
resp = requests.get(url)
resp.raise_for_status()

with open("model_quantized.onnx", "wb") as f:
    f.write(resp.content)
tokenizer = AutoTokenizer.from_pretrained("Xenova/multilingual-e5-small")
session = ort.InferenceSession("model_quantized.onnx")

def embed_query(q):
    inp = tokenizer([q], return_token_type_ids=True)
    outputs = session.run(["last_hidden_state"], dict(inp))
    emb = torch.tensor( outputs[0] ).float().mean(1)
    emb /= emb.norm(dim=-1)
    return emb[0]

Note: you may need to restart the kernel to use updated packages.


# 3. Λύνουμε το αντίστοιχο επίπεδο

In [20]:
answers = {}

In [21]:
levelid = 'level1' # 'level2' / 'level3' / 'level4'
level = levels[levelid]
target_order = torch.tensor([0,1,2,3,4])
E = embeddings[ level ] # embeddings ολων των ταινιών του επιπέδου
E.shape

torch.Size([15, 384])

In [22]:
import torch
import torch.nn.functional as F

# indices of the target permutation
targets = torch.tensor([0,1,2,3,4])


In [23]:
targets = target_order

T = E[targets]
others = E[[i for i in range(len(E)) if i not in targets]]

weights = torch.tensor([32.,16.,8.,4.,2.])

emb = (weights[:,None] * T).sum(dim=0)

# push other movies away
emb = emb - 4 * others.mean(dim=0)

emb = emb / emb.norm()

## Define Optimization Function

### Subtask:
Encapsulate the optimization logic for a single level into a reusable function, taking `level_data`, `global_embeddings`, and `target_order` as input, and returning the optimized `emb`.

### 🧠 Η Λογική της Βελτιστοποίησης (Optimization)

Σε αυτόν τον διαγωνισμό, στόχος μας είναι να βρούμε ένα διάνυσμα $q$ τέτοιο ώστε:
1. Οι ταινίες-στόχοι (targets) να έχουν την υψηλότερη ομοιότητα με το $q$.
2. Η σειρά τους να είναι ακριβώς $T_1 > T_2 > T_3 > T_4 > T_5$.
3. Κάθε ταινία $T_i$ να απέχει από την επόμενη $T_{i+1}$ τουλάχιστον κατά `min_diff`.
4. Όλες οι άλλες ταινίες του επιπέδου να έχουν σημαντικά μικρότερη ομοιότητα.

Χρησιμοποιούμε τη συνάρτηση $e^{-250 	imes (gap - threshold)}$ ως **Hinge Loss**. Όταν το $gap$ είναι μεγαλύτερο από το $threshold$, ο εκθέτης είναι αρνητικός και το loss σχεδόν μηδενικό. Όταν το $gap$ μικραίνει, το loss αυξάνεται εκθετικά!

In [24]:
T = E[:5]

weights = torch.tensor([16.,8.,4.,2.,1.])


In [25]:
import torch
import torch.nn.functional as F

def optimize_single_level(level_data, global_embeddings, target_order, min_diff=0.08, verbose=True):
    # --- 1. DATA PREPARATION ---
    # Retrieve the 384-dimensional embeddings for all movies in this specific level
    # We convert to float() to ensure high precision during gradient calculation
    E = global_embeddings[level_data].float()
    # Move the target indices (0-4) to the same device as E (e.g., GPU if available)
    targets = target_order.to(E.device)

    # T stores the high-dimensional vectors of our 5 target movies we want to rank
    T = E[targets]

    # Identify every movie in the level that is NOT part of our desired top 5
    # We will use these later to 'push' them away from our query vector
    all_indices = torch.arange(len(E), device=E.device)
    non_target_mask = torch.ones(len(E), dtype=torch.bool, device=E.device)
    non_target_mask[targets] = False
    non_target_indices = all_indices[non_target_mask]

    # --- 2. INITIALIZATION ---
    # We start 'q' as a weighted mixture of the target embeddings.
    # 'requires_grad_(True)' tells PyTorch that this is the variable we want to optimize.
    init_weights = torch.tensor([32., 16., 12., 4., 2.], device=E.device)
    q = (init_weights[:, None] * T).sum(0).clone().detach().requires_grad_(True)

    # Adam is an adaptive optimizer; it adjusts the learning rate for each dimension of q
    optimizer = torch.optim.Adam([q], lr=0.01)

    # gap_weights: We care extra about the 4th gap (Target 5 vs others) to ensure it stays 'on top'
    gap_weights = torch.tensor([1., 1., 1., 4.], device=E.device)

    best_loss = float('inf')
    patience = 500  # If the loss doesn't improve for 500 steps, we stop to save time
    no_improve = 0

    # --- 3. OPTIMIZATION LOOP ---
    # This loop 'nudges' the vector q step-by-step to satisfy our ranking constraints
    for step in range(80000):
        optimizer.zero_grad() # Reset the 'slopes' (gradients) from the last step

        # In vector similarity, the 'length' of the vector doesn't matter, only the 'angle'.
        # We normalize q to length 1 so that dot products equal cosine similarity.
        q_norm = q / q.norm()

        # Calculate the similarity score of 'q' against every movie in the level
        sims = F.cosine_similarity(E, q_norm.unsqueeze(0).expand_as(E))

        losses = []

        # PENALTY A: Ensuring Rank Order (T1 > T2 > T3 > T4 > T5)
        target_sims = sims[targets]
        for i in range(len(targets) - 1):
            # Gap = Similarity of movie at rank i MINUS similarity of movie at rank i+1
            gap = target_sims[i] - target_sims[i + 1]
            # HINGE LOSS: If gap < min_diff, the exponent is positive and loss becomes huge.
            # This 'punishes' the optimizer until the gap is large enough.
            losses.append(gap_weights[i] * torch.exp(-250 * (gap - min_diff)))

        # PENALTY B: Separation (All Targets > All Non-Targets)
        if len(non_target_indices) > 0:
            t_sims = sims[targets].unsqueeze(1) # [5, 1]
            n_sims = sims[non_target_indices].unsqueeze(0) # [1, others]
            # Compare every target against every non-target
            gaps = t_sims - n_sims
            # We penalize any non-target that gets too close to our targets
            losses.append(torch.exp(-250 * (gaps - min_diff)).sum())

        # Combine all individual penalties into one single number (The 'Mistake' score)
        loss = torch.stack(losses).sum()

        # This triggers the 'Auto-Diff' engine to calculate how to reduce the loss
        loss.backward()
        # The optimizer moves 'q' slightly in the direction that lowers the loss
        optimizer.step()

        if verbose and step % 2000 == 0:
            print(f"  Step {step}, Loss: {loss.item():.6f}")

        # EARLY STOPPING: Stop if we are no longer making meaningful progress
        if loss.item() < best_loss - 1e-6:
            best_loss = loss.item()
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"  Early stop at step {step}, Loss: {loss.item():.6f}")
                break

    # Final vector is the 'magic query' that ranks the movies correctly!
    return (q / q.norm()).detach()

In [26]:
# Ομοιότητα και Διάταξη ταινιών (φθίνουσα διάταξη με similarity)


In [27]:
# TODO: Βρείτε ένα query για κάθε επίπεδο ώστε οι πρώτες 5 ταινίες να είναι οι 0,1,2,3,4...


In [28]:
for levelid in levels:
  print("\n----------------")
  current_level_movie_ids = levels[levelid]
  target_order_within_level = torch.tensor([0,1,2,3,4]) # These are indices WITHIN the 'current_level_movie_ids' list

  emb = optimize_single_level(current_level_movie_ids, embeddings, target_order_within_level)

  # Calculate similarity with embeddings of movies in the current level
  level_embeddings = embeddings[current_level_movie_ids]
  similarity = torch.cosine_similarity(level_embeddings, emb)
  ranking = similarity.argsort(descending=True)

  print(f"Top 10 for {levelid}: {ranking[:10]}")
  print(f"Targets for {levelid}: {target_order_within_level}")
  is_correct_order = torch.all(ranking[:len(target_order_within_level)] == target_order_within_level).item()
  print(f"Is target order correct? {is_correct_order}")

  if is_correct_order:
      # Calculate sequential differences only if the order is correct
      diffs = -similarity[ranking].diff()[:len(target_order_within_level)]
      print('Sequential Differences {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}'.format(*diffs) )
      print('Minimum Difference {:.4f}'.format(diffs.min()))
  else:
      print("Target order is not correct, cannot calculate meaningful sequential differences.")

  answers[levelid] = {'embedding': emb.tolist()}


----------------
  Step 0, Loss: 31965868032.000000
  Step 2000, Loss: 30391112.000000
  Step 4000, Loss: 9158985.000000
  Step 6000, Loss: 3502129.000000
  Step 8000, Loss: 1455131.250000
  Step 10000, Loss: 626138.875000
  Step 12000, Loss: 274340.656250
  Step 14000, Loss: 121766.007812
  Step 16000, Loss: 54805.132812
  Step 18000, Loss: 25149.328125
  Step 20000, Loss: 11872.625000
  Step 22000, Loss: 5839.043945
  Step 24000, Loss: 3047.034424
  Step 26000, Loss: 1737.996704
  Step 28000, Loss: 1135.231689
  Step 30000, Loss: 893.941772
  Step 32000, Loss: 839.648315
  Step 34000, Loss: 837.479004
  Early stop at step 34853, Loss: 837.478271
Top 10 for level1: tensor([ 0,  1,  2,  3,  4,  5,  9, 10, 12,  7])
Targets for level1: tensor([0, 1, 2, 3, 4])
Is target order correct? True
Sequential Differences 0.0620, 0.0600, 0.0593, 0.0642, 0.0650
Minimum Difference 0.0593

----------------
  Step 0, Loss: 381864017920.000000
  Step 2000, Loss: 241359120.000000
  Step 4000, Loss: 8151

# 4. Αποθηκεύουμε τις απαντήσεις για υποβολή στο site

In [29]:
import json
with open('answers.json', 'w') as f:
    json.dump(answers, f)

In [31]:
# from google.colab import files

# files.download('answers.json')